# neural_optimiser Module

This module contains the core functionality for the Neural Network surrogat Process.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
import optunahub
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from .base import BaseBayesianOptimizer, Prediction

## SimpleNN

Defines a neural network with a specified number of layers, activation function, and dropout.  The forward function simply returns the output of the NN for a set of x values.
 

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, layer_sizes, activation_name, dropout):
        super().__init__()
        layers = []
        in_dim = input_dim
        
        activation_map = {
            'ReLU': nn.ReLU(), 'Tanh': nn.Tanh(), 'ELU': nn.ELU(), 'GELU': nn.GELU(), 'Sigmoid': nn.Sigmoid()
        }
        act_fn = activation_map.get(activation_name, nn.Tanh())

        for n_units in layer_sizes:
            layers.append(nn.Linear(in_dim, n_units))
            layers.append(act_fn) 
            if dropout > 1e-3: 
                layers.append(nn.Dropout(dropout))
            in_dim = n_units
            
        layers.append(nn.Linear(in_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x): return self.model(x)

# LivePlotCallback

Given training can take some time, this provides a continous plot of the latest results

In [1]:
class LivePlotCallback:
    def __init__(self, total_trials):
        self.best_val = float('inf')
        self.total = total_trials
        self.trial_nums = []
        self.mse_vals = []
        self.best_vals = []            
        self.fig, self.ax = plt.subplots(figsize=(10, 5))
        self.ax.set_xlabel('Trial')
        self.ax.set_ylabel('CV MSE')
        self.ax.grid(True, alpha=0.3)
        self.line_mse, = self.ax.plot([], [], 'o', color='grey', alpha=0.5)
        self.line_best, = self.ax.plot([], [], '-', color='red', linewidth=2)

    def __call__(self, study, trial):
        val = trial.value
        if val < self.best_val: self.best_val = val
        self.trial_nums.append(trial.number + 1)
        self.mse_vals.append(val)
        self.best_vals.append(self.best_val)
        
        self.line_mse.set_data(self.trial_nums, self.mse_vals)
        self.line_best.set_data(self.trial_nums, self.best_vals)
        self.ax.set_title(f'Neural Search - Best MSE: {self.best_val:.5f}')
        self.ax.relim()
        self.ax.autoscale_view()
        
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
        plt.pause(0.01)

    def close(self):
        plt.ioff()
        plt.close(self.fig)

## NeuralBayesianOptimizer

This inherits from the BaseBayesianOptimizer class in the base module and begins by applying standard scaling to the input.

### build_ensemble

Uses the optuna library to optimiser hyperparameter selection for the NN. create_study creates a optuna.study.Study object, which is the top-level orchestration engine and state manager for the hyperparameter search. This sends a optuna.trial.Trial object to the objective function. After running a defined number of trials, the top n_models are returned as an ensemble.

#### objective

The suggest methods send a request to the underlying Study object to return a set of hyperparameters based on the historical performance within the study. K-fold cross-validation is then used to train a NN with these hyperparameters and return the mean k-fold MSE.

### _train_models

This method trains the NN using the Adam optimiser.

### _predict_normalized

Returns the mean and std of the ensemble for a given set of x values.

In [ ]:
class NeuralBayesianOptimizer(BaseBayesianOptimizer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.x_scaler = StandardScaler()
        self.X_norm_nn = self.x_scaler.fit_transform(self.X)

    def build_ensemble(self, n_trials=100, n_models=5, patience=20) -> List[SimpleNN]:
        def objective(trial):
            n_layers = trial.suggest_int('n_layers', 2, 5)
            activation = trial.suggest_categorical('activation', ['Tanh', 'ReLU', 'GELU', 'ELU'])
            dropout = trial.suggest_float('dropout', 0.0, 0.05)
            lr = trial.suggest_float('lr', 5e-4, 1e-2, log=True)
            
            for i in range(n_layers):
                trial.suggest_int(f'n_units_l{i}', 32, 256)

            kf = KFold(n_splits=5, shuffle=True, random_state=self.seed)
            fold_scores = []
            
            for train_idx, val_idx in kf.split(self.X_norm_nn):
                model = self._train_model(trial.params, self.X_norm_nn[train_idx], self.y_norm[train_idx], epochs=100)
                model.eval()
                with torch.no_grad():
                    X_val = torch.FloatTensor(self.X_norm_nn[val_idx]).to(DEVICE)
                    y_val = torch.FloatTensor(self.y_norm[val_idx]).unsqueeze(1).to(DEVICE)
                    loss = nn.MSELoss()(model(X_val), y_val)
                fold_scores.append(loss.item())
            
            return np.mean(fold_scores)

        try:
            module = optunahub.load_module(package="samplers/auto_sampler")
            sampler = module.AutoSampler(seed=self.seed)
        except:
            from optuna.samplers import TPESampler
            sampler = TPESampler(seed=self.seed)

        study = optuna.create_study(direction='minimize', sampler=sampler)
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        
        live_plot = LivePlotCallback(n_trials)
        study.optimize(objective, n_trials=n_trials, callbacks=[live_plot])
        live_plot.close()

        models = []
        best_trials = sorted(study.trials, key=lambda t: t.value)[:n_models]
        for t in best_trials:
            m = self._train_model(t.params, self.X_norm_nn, self.y_norm, epochs=300)
            models.append(m)
        
        return models

    def _train_model(self, params, X, y, epochs):
        n_layers = params['n_layers']
        layer_sizes = [params[f'n_units_l{i}'] for i in range(n_layers)]
        model = SimpleNN(self.n_dims, layer_sizes, params['activation'], params['dropout']).to(DEVICE)
        opt = optim.Adam(model.parameters(), lr=params['lr'])
        crit = nn.MSELoss()
        
        Xt = torch.FloatTensor(X).to(DEVICE)
        yt = torch.FloatTensor(y).unsqueeze(1).to(DEVICE)
        
        model.train()
        for _ in range(epochs):
            opt.zero_grad()
            loss = crit(model(Xt), yt)
            loss.backward()
            opt.step()
        return model

    def _predict_normalized(self, model_list: List[SimpleNN], X_candidates: np.ndarray) -> Prediction:
        X_scaled = self.x_scaler.transform(X_candidates)
        Xt = torch.FloatTensor(X_scaled).to(DEVICE)
        preds = []
        for m in model_list:
            m.eval()
            with torch.no_grad():
                preds.append(m(Xt).cpu().numpy())
        
        preds = np.array(preds) # (n_models, n_samples, 1)
        mean_pred = np.mean(preds, axis=0).flatten()
        std_pred = np.std(preds, axis=0).flatten()
        return Prediction(mean=mean_pred, std=std_pred)